# Capítulo 7: El Post-Mortem (¿Qué Hizo Mal el Modelo y por Qué?)

> *"Un post-mortem no es una autopsia para castigar al muerto. Es una autopsia para que el siguiente paciente no muera igual."*

**Objetivo:** Diagnosticar por qué un modelo falla, aprender de los errores, y construir algo mejor. La métrica no es el accuracy, es el aprendizaje.

## Celda 1: Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, accuracy_score, precision_recall_curve,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    print('XGBoost:', xgb.__version__)
except ImportError:
    print('XGBoost no disponible')

try:
    from imblearn.over_sampling import SMOTE
    print('SMOTE disponible')
except ImportError:
    print('imblearn no disponible (pip install imbalanced-learn)')

pd.set_option('display.max_columns', 25)
np.random.seed(42)
print('\nLibrerías listas.')

## Celda 2: Carga de modelo entrenado

Recreamos el escenario: un modelo de predicción de rotación de empleados que "funciona" pero tiene problemas ocultos.

In [ ]:
df = pd.read_csv('../datos/datos_empleados_hr.csv')

print(f'Dataset: {df.shape[0]} empleados, {df.shape[1]} features')
print(f'\nDistribución de Attrition:')
print(df['Attrition'].value_counts())
print(f'\nTasa de rotación: {df["Attrition"].value_counts(normalize=True)["Yes"]*100:.1f}%')
print(f'\nClase mayoritaria (No): {df["Attrition"].value_counts(normalize=True)["No"]*100:.1f}%')
print(f'\n→ Si un modelo predice "No" para todos, tiene {df["Attrition"].value_counts(normalize=True)["No"]*100:.1f}% de accuracy.')
print(f'→ Ese es nuestro baseline inútil.')

In [ ]:
# Preprocesamiento
le_dict = {}
categorical_cols = ['Department', 'Gender', 'OverTime']
df_processed = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    le_dict[col] = le

df_processed['Attrition'] = (df_processed['Attrition'] == 'Yes').astype(int)

X = df_processed.drop('Attrition', axis=1)
y = df_processed['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')
print(f'Distribución train: {dict(y_train.value_counts())}')
print(f'Distribución test: {dict(y_test.value_counts())}')

In [ ]:
# Entrenar un modelo "malo" a propósito (sin balancear)
modelo_malo = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,   # sin límite → overfitting
    random_state=42,
    n_jobs=-1
)
modelo_malo.fit(X_train, y_train)

y_pred_malo = modelo_malo.predict(X_test)
y_proba_malo = modelo_malo.predict_proba(X_test)[:, 1]

print('=== Modelo Malo (sin ajustar) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_malo):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_malo):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_proba_malo):.4f}')
print(f'\n{classification_report(y_test, y_pred_malo, target_names=["No renuncia", "Sí renuncia"])}')

## Celda 3: Predicciones y probabilidades

Explorar las probabilidades predichas revela dónde el modelo tiene dudas.

In [ ]:
# Distribución de probabilidades predichas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probabilidades para clase real 0 (No renuncia)
axes[0].hist(y_proba_malo[y_test == 0], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
axes[0].set_title('Probabilidades predichas\nEmpleados que NO renunciaron', fontweight='bold')
axes[0].set_xlabel('Probabilidad predicha de renuncia')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Probabilidades para clase real 1 (Sí renuncia)
axes[1].hist(y_proba_malo[y_test == 1], bins=30, color='coral', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0.5, color='red', linestyle='--', label='Threshold=0.5')
axes[1].set_title('Probabilidades predichas\nEmpleados que SÍ renunciaron', fontweight='bold')
axes[1].set_xlabel('Probabilidad predicha de renuncia')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

# Análisis de confusiones
fp_mask = (y_pred_malo == 1) & (y_test == 0)
fn_mask = (y_pred_malo == 0) & (y_test == 1)

print(f'\nFalsos Positivos (predijo renuncia, no renunció): {fp_mask.sum()}')
print(f'Falsos Negativos (predijo no renuncia, sí renunció): {fn_mask.sum()}')
print(f'\n→ Los FN son los más peligrosos: perdimos al empleado sin aviso.')

## Celda 4: Matriz de confusión visual

La matriz de confusión es la tableta de verdad. No miente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz del modelo malo
cm_malo = confusion_matrix(y_test, y_pred_malo)
disp1 = ConfusionMatrixDisplay(cm_malo, display_labels=['No renuncia', 'Sí renuncia'])
disp1.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión\nModelo Malo (sin ajustar)', fontweight='bold')

# Modelo baseline: predecir "No" para todos
y_pred_baseline = np.zeros_like(y_test)
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
disp2 = ConfusionMatrixDisplay(cm_baseline, display_labels=['No renuncia', 'Sí renuncia'])
disp2.plot(ax=axes[1], cmap='Oranges', colorbar=False)
axes[1].set_title('Matriz de Confusión\nBaseline (todo "No")', fontweight='bold')

plt.tight_layout()
plt.show()

print('=== Análisis de la Matriz ===')
print(f'Modelo Malo: detecta {cm_malo[1,1]} de {cm_malo[1,0]+cm_malo[1,1]} renuncias (recall: {cm_malo[1,1]/(cm_malo[1,0]+cm_malo[1,1]):.1%})')
print(f'Baseline:    detecta {cm_baseline[1,1]} de {cm_baseline[1,0]+cm_baseline[1,1]} renuncias (recall: {cm_baseline[1,1]/(cm_baseline[1,0]+cm_baseline[1,1]):.1%})')
print(f'\n→ El modelo "malo" es apenas mejor que adivinar.')

## Celda 5: Precision, Recall, F1

¿Qué métrica importa? Depende del costo del error.

In [ ]:
# Tabla de métricas por modelo
modelos = {
    'Modelo Malo': (y_pred_malo, y_proba_malo),
    'Baseline (todo No)': (y_pred_baseline, np.zeros_like(y_test, dtype=float))
}

resultados = []
for nombre, (y_pred, y_proba) in modelos.items():
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Yes)': report['1']['precision'],
        'Recall (Yes)': report['1']['recall'],
        'F1 (Yes)': report['1']['f1-score'],
    })

df_metricas = pd.DataFrame(resultados).set_index('Modelo')
print('=== Métricas Comparativas ===')
print(df_metricas.round(4).to_string())

print('\n=== ¿Qué nos dice esto? ===')
print('Accuracy alto NO significa buen modelo.')
print('El baseline tiene 85% de accuracy y es completamente inútil.')
print('La métrica que importa es Recall (¿cuántas renuncias detecto?) y F1 (balance).')

## Celda 6: Curva ROC y AUC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# ROC del modelo malo
fpr, tpr, _ = roc_curve(y_test, y_proba_malo)
auc = roc_auc_score(y_test, y_proba_malo)
ax.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'Modelo Malo (AUC={auc:.3f})')

# Línea base (aleatorio)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio (AUC=0.500)')

# Línea perfecta
ax.plot([0, 0, 1], [0, 1, 1], 'g--', alpha=0.3, label='Perfecto (AUC=1.000)')

ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title('Curva ROC — Diagnóstico del Modelo', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

print(f'\nAUC = {auc:.4f}')
if auc >= 0.9:
    print('→ AUC alto. ¿Es demasiado bueno? sospechar de overfitting.')
elif auc >= 0.8:
    print('→ AUC aceptable. Hay espacio de mejora.')
elif auc >= 0.7:
    print('→ AUC mediocre. El modelo apenas supera el azar.')
else:
    print('→ AUC bajo. El modelo no es útil.')

In [ ]:
# Curva Precision-Recall (más informativa que ROC para clases desbalanceadas)
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(y_test, y_proba_malo)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_arr, precision_arr, color='coral', linewidth=2)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curva Precision-Recall\n(Más informativa que ROC para clases desbalanceadas)', fontweight='bold')
ax.axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline: {y_test.mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

print('\nLa curva PR muestra la verdad desnuda:')
print('A medida que el modelo intenta detectar más renuncias (recall sube),')
print('la precisión baja: empieza a confundir más.')

## Celda 7: Análisis de errores (muestras mal clasificadas)

¿Qué tienen en común los empleados que el modelo confundió?

In [ ]:
# Crear dataframe de errores
df_errores = X_test.copy()
df_errores['real'] = y_test.values
df_errores['predicho'] = y_pred_malo
df_errores['probabilidad'] = y_proba_malo

# Tipo de error
def clasificar_error(row):
    if row['real'] == 1 and row['predicho'] == 0:
        return 'FN (renunció, no detectó)'
    elif row['real'] == 0 and row['predicho'] == 1:
        return 'FP (no renunció, alertó)'
    elif row['real'] == 1 and row['predicho'] == 1:
        return 'VP (renunció, detectó)'
    else:
        return 'VN (no renunció, correcto)'

df_errores['tipo'] = df_errores.apply(clasificar_error, axis=1)

print('=== Distribución de Errores ===')
print(df_errores['tipo'].value_counts())

# Comparar perfiles: FN vs VP
fn_samples = df_errores[df_errores['tipo'] == 'FN (renunció, no detectó)']
vp_samples = df_errores[df_errores['tipo'] == 'VP (renunció, detectó)']

print(f'\n=== Perfil del Falso Negativo (el error más costoso) ===')
print(f'Empleados: {len(fn_samples)}')
print(f'\nCaracterísticas promedio de los FN:')
for col in ['MonthlyIncome', 'Age', 'YearsAtCompany', 'JobSatisfaction', 'WorkLifeBalance']:
    fn_mean = fn_samples[col].mean()
    vp_mean = vp_samples[col].mean()
    print(f'  {col:25s} FN={fn_mean:8.1f}  VP={vp_mean:8.1f}  (diff: {fn_mean-vp_mean:+.1f})')

In [ ]:
# Visualizar los errores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribución de probabilidades por tipo de error
for tipo, color in [('VP (renunció, detectó)', 'green'), 
                     ('FN (renunció, no detectó)', 'red'),
                     ('FP (no renunció, alertó)', 'orange')]:
    subset = df_errores[df_errores['tipo'] == tipo]
    if len(subset) > 0:
        axes[0].hist(subset['probabilidad'], bins=20, alpha=0.5, label=tipo, color=color)
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.5)
axes[0].set_title('Distribución de Probabilidades\npor Tipo de Resultado', fontweight='bold')
axes[0].set_xlabel('Probabilidad predicha')
axes[0].legend(fontsize=8)

# 2. MonthlyIncome vs Errores
for tipo, color, marker in [('VP (renunció, detectó)', 'green', 'o'), 
                              ('FN (renunció, no detectó)', 'red', 'X')]:
    subset = df_errores[df_errores['tipo'] == tipo]
    axes[1].scatter(subset['MonthlyIncome'], subset['probabilidad'], 
                    c=color, marker=marker, alpha=0.6, label=tipo, s=30)
axes[1].axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Ingreso vs Probabilidad\n(FN en rojo)', fontweight='bold')
axes[1].set_xlabel('MonthlyIncome')
axes[1].set_ylabel('Probabilidad predicha')
axes[1].legend(fontsize=8)

# 3. Top 10 errores más confiados (FN con alta probabilidad de "No")
fn_confidentes = df_errores[df_errores['tipo'] == 'FN (renunció, no detectó)'].nsmallest(10, 'probabilidad')
axes[2].barh(range(len(fn_confidentes)), fn_confidentes['probabilidad'], color='red', alpha=0.7)
axes[2].set_yticks(range(len(fn_confidentes)))
axes[2].set_yticklabels([f'Empleado {i}' for i in fn_confidentes.index])
axes[2].set_title('Top 10 FN más confiados\n(el modelo estaba seguro de que no renunciaban)', fontweight='bold')
axes[2].set_xlabel('Probabilidad predicha de NO renunciar')

plt.tight_layout()
plt.show()

## Celda 8: ¿Qué hizo mal el modelo? (Recuadro)

> **RECUADRO: ¿Qué hizo mal el modelo?**
>
>
> **Diagnóstico 1: Overfitting**
>
>> El modelo memorizó los patrones del entrenamiento en lugar de aprender generalizaciones. Es como estudiar para el examen memorizando las respuestas: sacas 10 en el simulacro pero 2 en el real.
>
> **Diagnóstico 2: Clases desbalanceadas**
>
>> Solo el 15% de los empleados renuncian. El modelo aprendió que lo más seguro es predecir "No". Es como un detector de humo que solo suena si hay un incendio de proporciones bíblicas.
>
> **Diagnóstico 3: Sin regularización**
>
>> `max_depth=None` permitió que los árboles crecieran sin control, aprendiendo ruido en vez de señal.
>
> **Diagnóstico 4: Features irrelevantes**
>
>> El modelo usó 23 features sin筛选. Algunas pueden estar confundiendo más que ayudando.
>
> **Diagnóstico 5: Threshold subóptimo**
>
>> El threshold de 0.5 no es el mejor para este problema. Para detectar renuncias, necesitamos un threshold más bajo.

In [ ]:
# Demostración: overfitting
print('=== Diagnóstico 1: Overfitting ===')
print(f'\nAccuracy en TRAIN: {accuracy_score(y_train, modelo_malo.predict(X_train)):.4f}')
print(f'Accuracy en TEST:  {accuracy_score(y_test, y_pred_malo):.4f}')
print(f'Gap:               {accuracy_score(y_train, modelo_malo.predict(X_train)) - accuracy_score(y_test, y_pred_malo):.4f}')

if accuracy_score(y_train, modelo_malo.predict(X_train)) - accuracy_score(y_test, y_pred_malo) > 0.05:
    print('\n→ Overfitting confirmado: el modelo memorizó el training set.')

print(f'\nProfundidad máxima de los árboles: {modelo_malo.estimators_[0].get_params()["max_depth"]}')
print('→ Sin límite de profundidad, cada árbol puede crear una regla para cada fila.')

In [ ]:
# Demostración: threshold subóptimo
print('\n=== Diagnóstico 5: Threshold Subóptimo ===')

precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_proba_malo)
f1_scores = 2 * (precision_arr * recall_arr) / (precision_arr + recall_arr + 1e-8)
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print(f'Threshold actual: 0.50')
print(f'Threshold óptimo: {best_threshold:.3f}')
print(f'\nCon threshold = 0.50:')
print(f'  Precision: {precision_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}')
print(f'  Recall:    {recall_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}')
print(f'\nCon threshold = {best_threshold:.3f}:')
print(f'  Precision: {precision_arr[best_idx]:.4f}')
print(f'  Recall:    {recall_arr[best_idx]:.4f}')
print(f'  F1:        {f1_scores[best_idx]:.4f}')

# Aplicar mejor threshold
y_pred_optimo = (y_proba_malo >= best_threshold).astype(int)
print(f'\nCon el threshold óptimo, el recall sube de {recall_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f} a {recall_arr[best_idx]:.4f}')
print(f'→ Detectamos más renuncias a costa de más falsas alarmas.')

## Celda 9: Plan de mejora

Un post-mortem sin plan de acción es solo una queja.

In [ ]:
print('='*60)
print('PLAN DE MEJORA — Modelo de Predicción de Rotación')
print('='*60)

print('\n## Paso 1: Balancear clases (impacto ALTO, esfuerzo BAJO)')
try:
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print(f'Después de SMOTE: {dict(pd.Series(y_train_bal).value_counts())}')
except NameError:
    print('SMOTE no disponible. Usando class_weight en su lugar.')
    X_train_bal, y_train_bal = X_train, y_train

print('\n## Paso 2: Regularización (impacto ALTO, esfuerzo BAJO)')
modelo_mejorado = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,              # limitar profundidad
    min_samples_split=20,      # mínimo para dividir
    min_samples_leaf=10,       # mínimo en hoja
    class_weight='balanced',   # compensar desbalance
    random_state=42,
    n_jobs=-1
)
modelo_mejorado.fit(X_train_bal, y_train_bal)

y_pred_mejorado = modelo_mejorado.predict(X_test)
y_proba_mejorado = modelo_mejorado.predict_proba(X_test)[:, 1]

print('\n## Paso 3: Ajustar threshold (impacto MEDIO, esfuerzo BAJO)')
precision_m, recall_m, thresholds_m = precision_recall_curve(y_test, y_proba_mejorado)
f1_m = 2 * (precision_m * recall_m) / (precision_m * recall_m + 1e-8)
best_t_m = thresholds_m[f1_m.argmax()]
y_pred_threshold = (y_proba_mejorado >= best_t_m).astype(int)

# Comparar
print('\n=== Resultados: Antes vs Después ===')
comparacion = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision (Yes)', 'Recall (Yes)', 'F1 (Yes)', 'AUC-ROC'],
    'Modelo Malo': [
        accuracy_score(y_test, y_pred_malo),
        classification_report(y_test, y_pred_malo, output_dict=True, zero_division=0)['1']['precision'],
        classification_report(y_test, y_pred_malo, output_dict=True, zero_division=0)['1']['recall'],
        f1_score(y_test, y_pred_malo),
        roc_auc_score(y_test, y_proba_malo)
    ],
    'Modelo Mejorado': [
        accuracy_score(y_test, y_pred_mejorado),
        classification_report(y_test, y_pred_mejorado, output_dict=True, zero_division=0)['1']['precision'],
        classification_report(y_test, y_pred_mejorado, output_dict=True, zero_division=0)['1']['recall'],
        f1_score(y_test, y_pred_mejorado),
        roc_auc_score(y_test, y_proba_mejorado)
    ],
    'Con Threshold Ajustado': [
        accuracy_score(y_test, y_pred_threshold),
        classification_report(y_test, y_pred_threshold, output_dict=True, zero_division=0)['1']['precision'],
        classification_report(y_test, y_pred_threshold, output_dict=True, zero_division=0)['1']['recall'],
        f1_score(y_test, y_pred_threshold),
        roc_auc_score(y_test, y_proba_mejorado)
    ]
}).set_index('Métrica')

print(comparacion.round(4).to_string())

In [ ]:
# Visualizar la mejora
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrices de confusión
cm_mejor = confusion_matrix(y_test, y_pred_mejorado)
disp1 = ConfusionMatrixDisplay(cm_malo, display_labels=['No', 'Yes'])
disp1.plot(ax=axes[0], cmap='Reds', colorbar=False)
axes[0].set_title('ANTES\nModelo Malo', fontweight='bold')

disp2 = ConfusionMatrixDisplay(cm_mejor, display_labels=['No', 'Yes'])
disp2.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('DESPUÉS\nModelo Mejorado', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nRenuncias detectadas: ANTES={cm_malo[1,1]} → DESPUÉS={cm_mejor[1,1]}')
print(f'Falsos negativos:     ANTES={cm_malo[1,0]} → DESPUÉS={cm_mejor[1,0]}')
print(f'→ Recuperamos {cm_mejor[1,1] - cm_malo[1,1]} empleados que hubiéramos perdido.')

## Celda 10: Ética — ¿El modelo discrimina?

El modelo "funciona". Pero ¿funciona igual para todos?

In [ ]:
print('='*60)
print('ANÁLISIS ÉTICO — ¿El modelo discrimina?')
print('='*60)

# Reconstuir dataset de test con género
df_test = X_test.copy()
df_test['Attrition_real'] = y_test.values
df_test['Attrition_pred'] = y_pred_mejorado
df_test['prob_renuncia'] = y_proba_mejorado

# Decodificar género
le_gender = le_dict['Gender']
df_test['Gender_label'] = le_gender.inverse_transform(df_test['Gender'].astype(int))

print('\n## 1. Tasa de rotación real por género')
rotacion_genero = df_test.groupby('Gender_label')['Attrition_real'].mean()
print(rotacion_genero.round(4))

print('\n## 2. Recall del modelo por género')
for gen in df_test['Gender_label'].unique():
    subset = df_test[df_test['Gender_label'] == gen]
    vp = ((subset['Attrition_real'] == 1) & (subset['Attrition_pred'] == 1)).sum()
    total_real = (subset['Attrition_real'] == 1).sum()
    recall = vp / total_real if total_real > 0 else 0
    print(f'  {gen}: {recall:.1%} de renuncias detectadas ({vp}/{total_real})')

print('\n## 3. Probabilidad promedio predicha por género')
prob_genero = df_test.groupby('Gender_label')['prob_renuncia'].mean()
print(prob_genero.round(4))

print('\n## 4. ¿Hay discriminación?')
recall_vals = []
for gen in df_test['Gender_label'].unique():
    subset = df_test[df_test['Gender_label'] == gen]
    vp = ((subset['Attrition_real'] == 1) & (subset['Attrition_pred'] == 1)).sum()
    total_real = (subset['Attrition_real'] == 1).sum()
    recall_vals.append(vp / total_real if total_real > 0 else 0)

diff_recall = max(recall_vals) - min(recall_vals)
if diff_recall > 0.10:
    print(f'  ⚠️  DIFERENCIA SIGNIFICATIVA: {diff_recall:.1%} de gap en recall entre géneros.')
    print(f'  → El modelo es menos preciso para un género que para otro.')
    print(f'  → Esto puede constituir discriminación algorítmica.')
elif diff_recall > 0.05:
    print(f'  ⚠️  Diferencia moderada: {diff_recall:.1%} de gap.')
    print(f'  → Monitorear en producción.')
else:
    print(f'  ✓ Diferencia menor: {diff_recall:.1%}. El modelo parece equilibrado.')

In [ ]:
# Visualizar公平idad
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Recall por género
recall_data = {}
for gen in df_test['Gender_label'].unique():
    subset = df_test[df_test['Gender_label'] == gen]
    vp = ((subset['Attrition_real'] == 1) & (subset['Attrition_pred'] == 1)).sum()
    total = (subset['Attrition_real'] == 1).sum()
    recall_data[gen] = vp / total if total > 0 else 0

axes[0].bar(recall_data.keys(), recall_data.values(), color=['steelblue', 'coral'])
axes[0].set_title('Recall por Género\n(¿Detecta igual las renuncias?)', fontweight='bold')
axes[0].set_ylabel('Recall')
axes[0].set_ylim(0, 1)
axes[0].axhline(y=np.mean(list(recall_data.values())), color='gray', linestyle='--', alpha=0.5)

# 2. Distribución de probabilidades por género
for gen, color in zip(df_test['Gender_label'].unique(), ['steelblue', 'coral']):
    subset = df_test[df_test['Gender_label'] == gen]
    axes[1].hist(subset['prob_renuncia'], bins=20, alpha=0.5, label=gen, color=color)
axes[1].set_title('Distribución de Probabilidades\npor Género', fontweight='bold')
axes[1].set_xlabel('Probabilidad predicha de renuncia')
axes[1].legend()

# 3. Ingreso por género (para contexto)
for gen, color in zip(df_test['Gender_label'].unique(), ['steelblue', 'coral']):
    subset = df_test[df_test['Gender_label'] == gen]
    axes[2].hist(subset['MonthlyIncome'], bins=20, alpha=0.5, label=gen, color=color)
axes[2].set_title('Distribución de Ingreso\npor Género (contexto)', fontweight='bold')
axes[2].set_xlabel('MonthlyIncome')
axes[2].legend()

plt.tight_layout()
plt.show()

print('\n=== RECOMENDACIONES ÉTICAS ===')
print('1. EXCLUIR Gender del modelo (variable protegida).')
print('2. Evaluar fairness por subgrupo DESPUÉS de entrenar.')
print('3. Documentar sesgos conocidos en la ficha técnica del modelo.')
print('4. Incluir a RRHH y Legal en la revisión del modelo.')
print('5. Monitorear métricas por subgrupo en producción.')
print('6. Establecer un proceso de apelación para empleados afectados.')

## Resumen del Capítulo

### Lo que aprendimos

1. **El accuracy miente.** Cuando las clases están desbalanceadas, un modelo inútil puede tener accuracy alto.
2. **La matriz de confusión es la tableta de verdad.** Te dice exactamente cómo falla el modelo.
3. **Los errores tienen patrones.** Analizar los falsos negativos revela qué se nos escapó.
4. **Un post-mortem sin plan es una queja.** Siempre documentar lecciones y acciones concretas.
5. **La ética no es opcional.** Un modelo "exitoso" que discrimina es un fracaso.

### La regla de oro

> *"No juzgues al modelo por su accuracy. Júzgalo por sus errores. Y cuando los entiendas, construye algo mejor."*

### Para el siguiente capítulo

Deep Learning: cuando los modelos de árboles no son suficientes, y necesitas redes neuronales para patrones más complejos.